## 02 Creando una API Key de Gemini

### Preparando la conexión con LLMs

In [ ]:
!pip install -q langchain langchain-google-genai google-generativeai

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

## 04 Usando Gemini por Langchain

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

Utilizando la API para obtener respuestas

In [ ]:
respuesta = llm.invoke("¿Qué es el RAG en Inteligencia Artificial?")

Explicando el concepto de RAG en inteligencia artificial

In [ ]:
respuesta.content

## 06 Creando el prompt de triaje

In [ ]:
PROMPT_TRIAJE = """
Eres un especialista en triaje del Service Desk para politicas internas.
Dado el mensaje del usuario, devuelve SÓLO un JSON con:\n
{\n
    "decision": "AUTO_RESOLVER" | "PEDIR_INFO" | "ABRIR_TICKET",\n
    "urgency": "BAJA" | "MEDIANA" | "ALTA",\n
    "missing_fields": ["..."]\n
}\n
Reglas:\n
- **AUTO_RESOLVER**: Preguntas claras sobre las reglas o procedimientos descritos en las politicas (Ej.: "¿Puedo reembolsar el internet para mi oficina en casa?").\n
- **PEDIR_INFO**: Mensajes imprecisos o sin información para identificar el tema o el contexto (Ej.: "Necesito ayuda con una politica").\n
- **ABRIR_TICKET**: Solicitudes de excepciones, autorización, aprobación o acceso especial, o cuando el usuario solicita explicitamente abrir un ticket (Ej.: "Quiero una excepción para trabajar remotamente durante 5 dias").\n
Analiza el mensaje y decide la acción más adecuada.
"""

Implementando la estructura de salida del agente

In [ ]:
from typing import Literal, List, Dict

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class TriajeOut(BaseModel):
    decision: Literal["AUTO_RESOLVER", "PEDIR_INFO", "ABRIR_TICKET"]
    urgencia: Literal["BAJA", "MEDIANA", "ALTA"]
    campos_faltantes: List[str] = Field(default_factory=list)

## 07 Cerebro del agente con structured output

In [ ]:
from langchain_core.messages import

In [ ]:
from langchain_core.messages import SystemMessage,

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

Configurando la cadena de triaje

In [ ]:
chain_de_triaje = llm.

In [ ]:
chain_de_triaje = llm.with_structured_output(TriajeOut)

Creando la función de triaje

In [ ]:
def triaje(mensaje: str) -> Dict:
    salida: TriajeOut = chain_de_triaje.invoke(
        [
            SystemMessage(content=PROMPT_TRIAJE),
            HumanMessage(content=mensaje)
        ]
    )
    return salida.model_dump()

Probando la función de triaje

In [ ]:
mensajes_de_prueba = [
    "¿Puedo obtener un reembolso por el internet de mi home office?",
    "Quiero una excepción para teletrabajar durante 5 días.",
    "¿Cómo funciona la política de comidas para viajes?",
    "¿Existe una política para anticipos de vacaciones?",
    "¿Quién fue Napoleón Bonaparte?"
]

In [ ]:
for pregunta in mensajes_de_prueba:
    r = triaje(pregunta)
    print(f"{pregunta} -> {r}")

# RAG

## 02 Cargando documentos para el RAG

In [ ]:
!pip install -q langchain_community faiss-cpu langchain-text-splitters pymupdf

Resolviendo problemas de instalación


In [ ]:
!pip install -q langchain_community==0.0.30 faiss-cpu==1.12.0 langchain-text-splitters==0.0.1 pymupdf==1.23.12

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for n in Path("/content/").glob("*.pdf"):
    try:
        loader = PyMuPDFLoader(str(n))
        docs.extend(loader.load())
        print(f"Archivo cargado: {n.name}")
    except Exception as e:
        print(f"Error cargando archivo: {n.name}: {e}")

print(f"Total de documentos cargados: {len(docs)}")

Preparando los documentos para el procesamiento


Al ejecutar este bloque, veremos que se han cargado los documentos: política de reembolsos, política de teletrabajo, y política de uso de correo electrónico, con un total de tres documentos cargados.

Para continuar, lo que haremos ahora es separar nuestros documentos, es decir, el texto dentro de cada documento, en algo que llamamos chunks (segmentos) en inteligencia artificial. Un chunk es una parte del documento. Lo que vamos a hacer en el rack es básicamente tomar la información que tenemos en este archivo PDF y enviarla al LLM. Sin embargo, no podemos enviar el texto tal cual está, porque antes tendremos que realizar una búsqueda sobre esta información, es decir, en qué documento y en qué parte de este documento se encuentra la información del prompt que deseamos.

Explicando el uso de embeddings

El modo en que hacemos esto en un rack es utilizando una técnica llamada Embeddings en inteligencia artificial. ¿Qué son los Embeddings? Básicamente, es un modo de convertir un texto, un segmento de texto, un chunk de texto (puede ser una imagen o un audio también, pero en nuestro caso es un texto) en un vector semántico del significado. Es decir, es un modo de convertir una parte del texto en un vector numérico. ¿Por qué? Porque los ordenadores no entienden texto, entienden números. Esta técnica permite convertir textos, imágenes o cualquier otro tipo de dato en un vector numérico que mantiene la parte semántica, es decir, el significado del texto comparado con otras cosas.

Por ejemplo, podemos convertir la parte "gestión de contraseñas" en un vector numérico. Existen modelos de inteligencia artificial, llamados modelos de Embedding, que saben convertir todo este texto en un vector numérico. Este vector puede tener muchas o pocas dimensiones, dependiendo del modelo. Puede tener 300 dimensiones, que es un número bastante normal, o 100, que es un número más bajo. Podemos ver esto en el MTAB Leaderboard de Hugging Face, una plataforma muy popular de modelos Open Source de inteligencia artificial y datasets. El MTAB Leaderboard muestra los mejores modelos de Embedding, que crean estos Embeddings. Por ejemplo, el modelo Gemini Embedding 001 tiene 3,072 dimensiones, es decir, un vector con hasta 3,072 números. Otro modelo, el Embedding Gemma 300 millones, tiene 768 dimensiones. La parte importante es que saben cómo convertir esto en números manteniendo los conceptos semánticos o el significado.

Realizando operaciones matemáticas con embeddings

Por eso, sabemos que la palabra "perro" estará muy cerca de la palabra "gato" o "ratón". Este modelo sabe crear un entorno de Embeddings. Por ejemplo, en una imagen podemos ver conceptos como "pájaro", "perro", "gato", que están más o menos cercanos, y otros como "elefante", "jirafa", "tortuga", que están un poco más lejos. Los creadores de este concepto de Embeddings han descubierto que podemos realizar operaciones matemáticas con ellos. Por ejemplo, podemos tomar el concepto "rey", que también es un vector numérico, y restar el vector de "hombre" y sumar el vector de "mujer". El resultado de esta operación matemática de vectores sería cercano al vector de la palabra "reina". Así, podemos realizar operaciones matemáticas con significados de palabras, textos y segmentos. Es algo realmente fantástico.

Separando documentos en chunks


En la próxima parte, haremos exactamente esto: separar nuestros PDFs en chunks, en segmentos menores, y crear un vector numérico que representará cada chunk. Luego, compararemos con el vector de la pregunta. Podremos convertir la pregunta en un vector numérico y compararla con todos los chunks, para identificar cuáles están más cercanos y cuáles más lejanos. El más cercano probablemente será la respuesta correcta a la pregunta que haremos al modelo. Esto es básicamente cómo funciona un RAG por detrás, con este concepto de embeddings muy fuerte.

Para terminar este video, lo que haremos es exactamente esto: separar nuestros archivos en chunks específicos. Para ello, utilizaremos una biblioteca. Importaremos el recursive_character_text_splitter y crearemos nuestro separador utilizando este recursive_character_text_splitter.

In [ ]:
from langchain.text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

Visualizando los chunks generados


In [ ]:
docs_splits = splitter.split_documents(docs)

In [ ]:
for chunk in docs_splits:
    print(chunk)
    print("-----------------")

## 03 Configurando embeddings y vectorstore

Importando y configurando la clase de embeddings

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
modelo_embeddings = GoogleGenerativeAIEmbeddings()

In [ ]:
modelo_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

Cargando la VectorStore


In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vectorstore = FAISS.from_documents(chunks, modelo_embeddings)

Creando el retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3}
)

Ajustando parámetros del retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 4}
)

## 05 Prompt del RAG

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

Configurando el prompt para el RAG


In [ ]:
prompt_rag = ChatPromptTemplate(
    [
        ("system",
            """Eres el especialista en RR.HH. de la empresa Carraro Desarrollo de Software.
            Responde siempre utilizando los conocimientos de las bases de datos pasadas a ti.
            Si no hay informacion sobre la pregunta en los datos, responde solo 'No lo se'.
            """
        ),
        ("human", "Contexto: {context}\nPregunta del empleado: {input}")
    ]
)

Creando la cadena de documentos


In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt_rag)

## 06 Funcion del RAG

Definiendo la estructura de la función


In [ ]:
def busqueda_de_respuestas_RAG(pregunta) -> Dict:
    documentos_relacionados = retriver.invoke(pregunta)

In [ ]:
{
    "respuesta": str,
    "citaciones": [],
    "docuemntos_encontrados": bool
}


## 08 Ejecutando el RAG

In [ ]:
busqueda_de_respuestas_RAG()

In [ ]:
mensajes_de_prueba = [
    "¿Puedo obtener un reembolso por el internet de mi home office?",
    "Quiero una excepción para teletrabajar durante 5 días.",
    "¿Cómo funciona la política de comidas para viajes?",
    "¿Existe una politica para anticipos de vacaciones?",
    "¿Quién fue Napoleon Bonaparte?"
]

In [ ]:
busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")

In [ ]:
r = busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")

In [ ]:
r = busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")
print(r)

In [ ]:
len(r["citaciones"])

Iterando sobre las preguntas de prueba


In [ ]:
for pregunta in mensajes_de_prueba:
    respuesta_RAG = busqueda_de_respuestas_RAG(pregunta)
    print(f"PREGUNTA: {pregunta}")

Imprimiendo respuestas y citaciones


In [ ]:
if respuesta_RAG['documentos_encontrados']:
    for i, citacion in enumerate(respuesta_RAG['citaciones']):
        print(f"CITACION {i + 1}:")
        print(f"Camino del documento: {citacion.metadata['file_path']}")
        print(f"Contenido: {citacion.page_content.replace('\n', ' ')}")
print("-------------------")